In [1]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 3.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [boto3]32m3/4 [boto3]sfer]


In [7]:
# INI LATIHAN, KODENYA CARI SENDIRI DARI AI
# langkah 1 koneksikan/hubungkan dengan data.lake
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://seaweedfs:8333",
    aws_access_key_id='admin',
    aws_secret_access_key='secret',
)

In [13]:
# Tampilkan daftar nama bucket
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(f"- {bucket['Name']}")

- bronze
- gold
- silver


In [14]:
# Cuma pastiin sudah terhubung dengan data lake atau blm -- cttn dari AI
response = s3.list_buckets()

print(response)

{'ResponseMetadata': {'RequestId': '18DB34A332D4388EB2A0D0CB', 'HostId': '', 'HTTPStatusCode': 200, 'HTTPHeaders': {'accept-ranges': 'bytes', 'content-length': '481', 'content-type': 'application/xml', 'server': 'SeaweedFS 30GB 4.48', 'x-amz-request-id': '18DB34A332D4388EB2A0D0CB', 'date': 'Sun, 04 Oct 2026 03:05:52 GMT'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'bronze', 'CreationDate': datetime.datetime(2026, 10, 4, 3, 4, 31, tzinfo=tzlocal())}, {'Name': 'gold', 'CreationDate': datetime.datetime(2026, 10, 4, 3, 4, 32, tzinfo=tzlocal())}, {'Name': 'silver', 'CreationDate': datetime.datetime(2026, 10, 4, 3, 4, 32, tzinfo=tzlocal())}], 'Owner': {'DisplayName': 'admin-admin', 'ID': 'admin-admin'}}


In [12]:
# Buat bucket dengan nama bronze, silver, gold
s3.create_bucket(Bucket='bronze')
s3.create_bucket(Bucket='silver')
s3.create_bucket(Bucket='gold')

{'ResponseMetadata': {'RequestId': '18DB349098843249478FFB7E',
  'HostId': '',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'location': '/gold',
   'server': 'SeaweedFS 30GB 4.48',
   'vary': 'Origin',
   'x-amz-request-id': '18DB349098843249478FFB7E',
   'date': 'Sun, 04 Oct 2026 03:04:32 GMT'},
  'RetryAttempts': 0},
 'Location': '/gold'}

In [15]:
# Upload sales.csv ke bucket bronze
s3.upload_file(
    "sales.csv",
    "bronze",
    "sales.csv"
)

In [16]:
# Tampilkan isi bucket bronze (cttan dari AI)

response = s3.list_objects_v2(Bucket="bronze")

for obj in response.get("Contents", []):
    print(f"- {obj['Key']}")

- sales.csv


In [18]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 4.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 3.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [19]:
# bersihkan data sales.csv lalu simpan ke bucket silver dengan nama sales_cleaned.csv
import pandas as pd
import io

# Membaca sales.csv dari bucket bronze
response = s3.get_object(
    Bucket="bronze",
    Key="sales.csv"
)

# Membaca data CSV
df = pd.read_csv(response["Body"], sep=";")

# Membersihkan data
df = df[(df["harga"] > 0) & (df["jumlah"] > 0)]

# Mengubah kembali menjadi CSV
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False, sep=";")

# Menyimpan data yang sudah dibersihkan ke bucket silver
s3.put_object(
    Bucket="silver",
    Key="sales_cleaned.csv",
    Body=csv_buffer.getvalue(),
    ContentType="text/csv"
)

print("Data sales berhasil dibersihkan dan disimpan ke bucket silver.")

Data sales berhasil dibersihkan dan disimpan ke bucket silver.


In [20]:
# Cek bucket silver --cttn dr Ai
response = s3.list_objects_v2(Bucket="silver")

for obj in response.get("Contents", []):
    print(f"- {obj['Key']}")

- sales_cleaned.csv


In [21]:
# aggregate data sales_cleaned.csv lalu simpan ke bucket gold dan tampilkan
# aggregate data sales_cleaned.csv lalu simpan ke bucket gold dan tampilkan

import pandas as pd
import io

# Membaca sales_cleaned.csv dari bucket silver
response = s3.get_object(
    Bucket="silver",
    Key="sales_cleaned.csv"
)

# Membaca CSV
df = pd.read_csv(response["Body"], sep=";")

# Menghitung total penjualan
df["total_penjualan"] = df["harga"] * df["jumlah"]

# Aggregate berdasarkan produk
aggregate = df.groupby("produk").agg(
    total_jumlah=("jumlah", "sum"),
    total_penjualan=("total_penjualan", "sum")
).reset_index()

# Tampilkan hasil aggregate
print(aggregate)

# Mengubah hasil aggregate menjadi CSV
csv_buffer = io.StringIO()
aggregate.to_csv(csv_buffer, index=False, sep=";")

# Simpan ke bucket gold
s3.put_object(
    Bucket="gold",
    Key="sales_aggregate.csv",
    Body=csv_buffer.getvalue(),
    ContentType="text/csv"
)

print("\nData aggregate berhasil disimpan ke bucket gold sebagai sales_aggregate.csv")

  produk  total_jumlah  total_penjualan
0     p1             2             2000
1     p2             3             6000
2     p3             6             8400

Data aggregate berhasil disimpan ke bucket gold sebagai sales_aggregate.csv
